In [ ]:
from pathlib import Path
import sys
sys.path.append("../benchmark/")
from parse_json import *

gt_path = Path("/media/EVO870/datasets/SNP_Summer_2024/experiments/Data_Processing_2024/annotated_tracklets_json/")

In [2]:
gt_files = list(gt_path.rglob("*.json"))

In [3]:
def check_file_gt(json_file) -> bool:
    video_detections = load_video_detections(json_file)
    individual_tracks = get_tracks_from_video_detections(video_detections)

    return video_contains_species(individual_tracks, "wolf")

def check_file_gt(json_file) -> bool:
    video_detections = load_video_detections(json_file)
    individual_tracks = get_tracks_from_video_detections(video_detections)

    adult_red_deer_tracks = get_deer_tracks_from_age(individual_tracks, "adult")

    if len(adult_red_deer_tracks) != 2:
        return False

    deer_grazing = get_tracks_from_action(adult_red_deer_tracks, "grazing")
    deer_foraging = get_tracks_from_activity(adult_red_deer_tracks, "foraging")

    return (len(deer_grazing) == 2 and  len(deer_foraging) == 2)

In [4]:
def check_file_qwen(json_file) -> bool:
    video_detections = load_video_detections(json_file)
    individual_tracks = get_tracks_from_video_detections(video_detections)
    species_result = video_contains_species(individual_tracks, species_name="red_deer", min_occurences=2)
    age_result = video_contains_deer_age(individual_tracks, age="adult", min_occurences=2)
    action_result = video_contains_action(individual_tracks, action_name="grazing", min_occurences=1)
    activity_result = video_contains_activity(individual_tracks, activity_name="foraging", min_occurences=1)
    return species_result and age_result and action_result and activity_result

In [5]:
def check_file_mistral(json_file) -> bool:
    video_detections = load_video_detections(json_file)
    individual_tracks = get_tracks_from_video_detections(video_detections)
    return video_contains_species(individual_tracks, "wolf")

In [6]:
def check_file_apertus(json_file) -> bool:
    video_detections = load_video_detections(json_file)
    individual_tracks = get_tracks_from_video_detections(video_detections)

    # Check if the video contains red deer tracks
    if video_contains_species(individual_tracks, 'red_deer', min_occurences=1):
        return True

    # Check if the video contains tracks of adult red deer
    if video_contains_adult_deer_sex(individual_tracks, 'adult', min_occurences=1):
        return True

    # Check if the video contains tracks of red deer grazing while foraging
    if video_contains_action(individual_tracks, 'grazing', min_occurences=1) and video_contains_activity(individual_tracks, 'foraging', min_occurences=1):
        return True

    return False

In [ ]:
def check_file_llama(json_file) -> bool:
    video_detections = load_video_detections(json_file)
    individual_tracks = get_tracks_from_video_detections(video_detections)
    deer_tracks = get_tracks_from_species(individual_tracks, 'Red Deer')
    grazing_tracks = get_tracks_from_action(deer_tracks, 'Grazing')
    foraging_tracks = get_tracks_from_activity(grazing_tracks, 'Foraging')

    # Check if all tracks match the prompt
    return len(foraging_tracks) == len(deer_tracks) == len(grazing_tracks) == len(individual_tracks)

In [8]:
matching_files = set()
for json_file in gt_files:
    if check_file_gt(json_file):
        matching_files.add(json_file)

In [9]:
len(matching_files)

7

In [16]:
retrieved_files = set()
for json_file in gt_files:
    if check_file_llama(json_file):
        retrieved_files.add(json_file)

In [17]:
len(retrieved_files)

533

In [18]:
len(retrieved_files & matching_files)

7